# Multi-Agent Emergency Response System
### Agent Dispatch Module - Real Dataset Integration
**Manya Mahajan** · AI Agent Developer

---

### Project Overview

This notebook implements the **Agent Dispatch Module** of a multi-agent emergency response system designed for the state of Victoria, Australia. The system automates the coordination of three emergency service types - ambulance, fire, and police by intelligently determining which agents to dispatch, how many units are required, and which real-world facility is closest to the incident.

Rather than relying on hardcoded placeholder data, this implementation integrates verified Victorian geographic datasets. Each agent resolves its nearest available facility at dispatch time using Haversine distance calculations against real hospital, fire station, and police station coordinates. 

### User Stories

The dispatch module addresses the following user stories:

- **As an emergency dispatcher**, I want the system to automatically identify the correct combination of services (ambulance, fire, police) for each emergency type, so that the appropriate resources are mobilised without manual lookup.
- **As an emergency dispatcher**, I want each dispatched agent to report the nearest real facility and its distance from the incident, so that I can verify response coverage and estimated arrival.
- **As a system operator**, I want the dispatch logic to gracefully handle unavailable units by flagging them in the incident report rather than failing silently, so that resource gaps are immediately visible.
- **As the LLM integration developer**, I want a clean `dispatch()` interface that accepts an emergency type, location string, and coordinates and returns a structured incident report, so that my natural language parsing layer can call it directly without understanding the internal agent logic.
- **As the routing engine developer**, I want every agent response to include a clearly defined `route` field and the incident coordinates, so that I can plug my routing graph (built from `road_nodes.csv` and `road_edges.csv`) into the dispatch pipeline without modifying the agent classes.
- **As the dataset team**, I want the system to consume standardised CSVs with `lat`/`lon` columns through a single `find_nearest()` interface, so that we can supply updated or additional facility datasets without requiring code changes.
- **As a project stakeholder**, I want the system to be validated against real Victorian incident data with a clear severity-to-emergency mapping, so that I can be confident the dispatch logic performs correctly under realistic conditions.

### Datasets

| Dataset | Purpose |
|---|---|
| `hospitals.csv` | Nearest hospital lookup for ambulance dispatch |
| `fire_stations.csv` | Nearest station lookup for fire dispatch |
| `police_stations.csv` | Nearest station lookup for police dispatch |
| `emergency_crash.csv` | Real Victorian crash records for system validation |
| `road_nodes.csv` | Road network graph - routing engine handoff |
| `road_edges.csv` | Road network graph - routing engine handoff |
| `cleaned_transport_2025 (2).csv` | Public transport network data |
| `pedestrian_data.csv` | Pedestrian activity data |

---
## 1 · Load and Validate Datasets

Before any dispatch logic can run, the system needs reliable geographic data. This cell loads all eight CSV datasets into pandas DataFrames, standardises column names across sources (each dataset uses slightly different conventions), converts coordinates to numeric types, and filters the hospital dataset down to only those facilities with emergency department capability.

A quality check is then run across every dataset to confirm row counts, column names, null values, and where applicable, coordinate ranges. These ranges are expected to fall within Victoria's geographic bounds (approximately −39° to −34° latitude, 141° to 150° longitude), giving us confidence that the data has been correctly sourced and parsed before it is used in distance calculations downstream.

In [3]:
import pandas as pd
import os
import math
import warnings
from datetime import datetime
import zipfile, io, requests

warnings.filterwarnings('ignore')

# File paths
base_url = "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/Project4_Multi_Agent_Emergency_Response_System_Datasets"

hospitals       = pd.read_csv(f"{base_url}/hospitals.csv")
fire_stations   = pd.read_csv(f"{base_url}/fire_stations.csv")
police_stations = pd.read_csv(f"{base_url}/police_stations.csv")
crash_data      = pd.read_csv(f"{base_url}/emergency_crash.csv")
road_nodes      = pd.read_csv(f"{base_url}/road_nodes.csv")
transport_data  = pd.read_csv(f"{base_url}/cleaned_transport_2025%20(2).csv")
pedestrian_data = pd.read_csv(f"{base_url}/pedestrian_data.csv")

r = requests.get(f"{base_url}/road_edges.zip")
z = zipfile.ZipFile(io.BytesIO(r.content))
road_edges = pd.read_csv(z.open(z.namelist()[0]))

# Clean column names
all_datasets = [hospitals, fire_stations, police_stations, crash_data,
                road_nodes, road_edges, transport_data, pedestrian_data]

for df in all_datasets:
    df.columns = df.columns.str.strip()

# Standardise lat/lon column names
for df in [hospitals, fire_stations, police_stations, crash_data, road_nodes]:
    df.rename(columns={'latitude': 'lat', 'longitude': 'lon'}, inplace=True)

# Convert coordinates to float
for df in [hospitals, fire_stations, police_stations, crash_data, road_nodes]:
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

# Filter hospitals to emergency-capable only
emergency_hospitals = hospitals[
    hospitals['emergency'].str.lower() == 'yes'
].copy().reset_index(drop=True)

# Drop rows with missing coordinates
police_stations = police_stations.dropna(subset=['lat', 'lon']).reset_index(drop=True)

# Quality check - coordinate datasets
coord_datasets = {
    'Emergency hospitals' : emergency_hospitals,
    'Fire stations'       : fire_stations,
    'Police stations'     : police_stations,
    'Crash incidents'     : crash_data,
    'Road nodes'          : road_nodes,
}

print("=" * 65)
print("  DATASET QUALITY CHECK")
print("=" * 65)

for name, df in coord_datasets.items():
    nulls   = df[['lat', 'lon']].isnull().sum().sum()
    lat_rng = f"{df['lat'].min():.2f} → {df['lat'].max():.2f}"
    lon_rng = f"{df['lon'].min():.2f} → {df['lon'].max():.2f}"
    print(f"\n  {name}")
    print(f"    Rows        : {len(df):,}")
    print(f"    Null coords : {nulls}")
    print(f"    Lat range   : {lat_rng}")
    print(f"    Lon range   : {lon_rng}")

# Road edges
total_length = road_edges['length'].sum() if 'length' in road_edges.columns else None
print(f"\n  Road edges")
print(f"    Rows        : {len(road_edges):,}")
if total_length:
    print(f"    Total length: {total_length:,.1f} metres")

# Transport data
print(f"\n  Transport data")
print(f"    Rows        : {len(transport_data):,}")
print(f"    Columns     : {list(transport_data.columns)}")
print(f"    Null values : {transport_data.isnull().sum().sum()}")

# Pedestrian data
print(f"\n  Pedestrian data")
print(f"    Rows        : {len(pedestrian_data):,}")
print(f"    Columns     : {list(pedestrian_data.columns)}")
print(f"    Null values : {pedestrian_data.isnull().sum().sum()}")

# Summary
print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Emergency hospitals  : {len(emergency_hospitals)}")
print(f"  Fire stations        : {len(fire_stations)}")
print(f"  Police stations      : {len(police_stations)}")
print(f"  Crash incidents      : {len(crash_data):,}")
print(f"  Road nodes           : {len(road_nodes):,}")
print(f"  Road edges           : {len(road_edges):,}")
print(f"  Transport records    : {len(transport_data):,}")
print(f"  Pedestrian records   : {len(pedestrian_data):,}")
print("=" * 65)

  DATASET QUALITY CHECK

  Emergency hospitals
    Rows        : 26
    Null coords : 0
    Lat range   : -38.36 → -37.65
    Lon range   : 144.70 → 145.35

  Fire stations
    Rows        : 205
    Null coords : 0
    Lat range   : -38.43 → -37.50
    Lon range   : 144.50 → 145.75

  Police stations
    Rows        : 124
    Null coords : 0
    Lat range   : -38.37 → -37.51
    Lon range   : 144.58 → 145.72

  Crash incidents
    Rows        : 194,352
    Null coords : 0
    Lat range   : -39.03 → -34.12
    Lon range   : 140.97 → 149.76

  Road nodes
    Rows        : 228,213
    Null coords : 0
    Lat range   : -38.49 → -37.44
    Lon range   : 144.46 → 145.91

  Road edges
    Rows        : 501,205
    Total length: 59,139,312.0 metres

  Transport data
    Rows        : 258,573
    Columns     : ['id', 'location_name', 'Latitude', 'Longitude', 'date', 'class', 'count']
    Null values : 540

  Pedestrian data
    Rows        : 734,064
    Columns     : ['sensor_id', 'date', 'hour

**Results:** All 8 datasets loaded and validated successfully.

**Service facility coverage** spans the greater Melbourne and surrounding Victorian region with zero null coordinates across all five coordinate-bearing datasets:
- **Emergency hospitals** - 26 facilities, latitude −38.36 to −37.65, longitude 144.70 to 145.35
- **Fire stations** - 205 stations, latitude −38.43 to −37.50, longitude 144.50 to 145.75
- **Police stations** - 124 stations, latitude −38.37 to −37.51, longitude 144.58 to 145.72

**Crash incident data** covers a much wider area across Victoria (194,352 records), ranging from latitude −39.03 to −34.12 and longitude 140.97 to 149.76 — confirming statewide coverage well beyond the metropolitan facility boundaries.

**Road network** consists of 228,213 nodes and 501,205 edges with a total road length of approximately 59,139 km, ready for handoff to Basil's routing engine in Sprint 2.

**Transport data** contains 258,573 records across 7 columns including location coordinates, date, transport class, and passenger counts. 540 null values were detected and will need to be handled depending on downstream usage.

**Pedestrian data** is the largest dataset at 734,064 records with zero null values. It includes hourly pedestrian counts by sensor, congestion scores, and peak hour flags, useful for understanding foot traffic density around incident locations.

---
## 2 · Emergency Types and Dispatch Table

The dispatch table is the decision-making backbone of the system. It defines every emergency type the system recognises, maps each to the combination of agents that should respond, and assigns a priority level (1 = critical, 2 = high, 3 = medium). The central dispatch function in Section 6 references this table at runtime to determine which agents to activate for any given incident.

In [6]:
# Emergency Types & Dispatch Table

DISPATCH_TABLE = {
    "cardiac_arrest": {
        "agents": ["ambulance"],
        "priority": 1,
        "notes": "Medical emergency - ambulance only"
    },
    "fire": {
        "agents": ["fire", "ambulance"],
        "priority": 1,
        "notes": "Fire service leads, ambulance on standby for injuries"
    },
    "car_accident_minor": {
        "agents": ["ambulance", "police"],
        "priority": 2,
        "notes": "Ambulance for injuries, police for traffic control"
    },
    "car_accident_major": {
        "agents": ["ambulance", "fire", "police"],
        "priority": 1,
        "notes": "All services — entrapment likely, major injuries expected"
    },
    "robbery": {
        "agents": ["police"],
        "priority": 2,
        "notes": "Police only"
    },
    "assault": {
        "agents": ["police", "ambulance"],
        "priority": 2,
        "notes": "Police to secure scene, ambulance for victim"
    },
    "building_collapse": {
        "agents": ["fire", "ambulance", "police"],
        "priority": 1,
        "notes": "All services — search and rescue situation"
    },
    "gas_leak": {
        "agents": ["fire", "police"],
        "priority": 1,
        "notes": "Fire handles hazard, police for evacuation"
    },
    "drowning": {
        "agents": ["ambulance", "police"],
        "priority": 1,
        "notes": "Ambulance for resuscitation, police for scene control"
    },
    "unknown": {
        "agents": ["police"],
        "priority": 3,
        "notes": "Default - police assess and call in other services if needed"
    }
}

print(f"{'EMERGENCY TYPE':<25} {'AGENTS':<35} {'PRI'}")
print("─" * 65)
for etype, info in DISPATCH_TABLE.items():
    print(f"{etype:<25} {', '.join(info['agents']):<35} {info['priority']}")

EMERGENCY TYPE            AGENTS                              PRI
─────────────────────────────────────────────────────────────────
cardiac_arrest            ambulance                           1
fire                      fire, ambulance                     1
car_accident_minor        ambulance, police                   2
car_accident_major        ambulance, fire, police             1
robbery                   police                              2
assault                   police, ambulance                   2
building_collapse         fire, ambulance, police             1
gas_leak                  fire, police                        1
drowning                  ambulance, police                   1
unknown                   police                              3


**Results:** The dispatch table defines 10 emergency types across three priority levels. Six are classified as critical (priority 1): cardiac arrest, fire, major car accident, building collapse, gas leak, and drowning. Three are high priority (priority 2): minor car accident, robbery, and assault. One is medium priority (priority 3): unknown emergencies, which default to a police-first response for on scene assessment.

Agent combinations range from single-service responses (cardiac arrest dispatches ambulance only, robbery dispatches police only) to full three-service mobilisation for major incidents like building collapses and major car accidents. This ensures resource allocation scales proportionally to incident severity. The `unknown` type acts as a catch-all - police are sent first to assess the situation and can request additional services if needed.

---
## 3 · Nearest-Unit Lookup (`find_nearest`)

Every agent needs to answer one question: *given an incident at these coordinates, which is the closest facility to dispatch from?*

`find_nearest()` uses the **Haversine formula** to calculate great-circle distances between the incident location and every candidate facility in a given dataset, then returns the closest match. A companion function `find_nearest_n()` returns the top *n* closest facilities, which is useful for identifying backup units.

This single interface is deliberately dataset-agnostic, it works with any DataFrame containing `lat`, `lon`, and a name column. This means the dataset team can supply updated or additional facility CSVs without requiring any code changes to the agents.

In [9]:

# Haversine Distance & Nearest-Unit Functions

def haversine(lat1, lon1, lat2, lon2):
    """
    Great-circle distance between two lat/lon points.
    Returns distance in kilometres.
    """
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat / 2) ** 2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def find_nearest(incident_lat, incident_lon, locations_df, name_col='name'):
    """
    Returns the single closest location from a DataFrame.

    Parameters:
    
    incident_lat, incident_lon : float
        Coordinates of the incident.
    locations_df : DataFrame
        Must contain 'lat', 'lon', and `name_col` columns.
    name_col : str
        Column used as the facility name in the result.

    Returns
    
    dict : name, lat, lon, distance_km
    """
    df = locations_df.copy()
    df['distance_km'] = df.apply(
        lambda row: haversine(incident_lat, incident_lon, row['lat'], row['lon']),
        axis=1
    )
    nearest = df.loc[df['distance_km'].idxmin()]
    return {
        'name'       : nearest[name_col],
        'lat'        : nearest['lat'],
        'lon'        : nearest['lon'],
        'distance_km': round(nearest['distance_km'], 3)
    }


def find_nearest_n(incident_lat, incident_lon, locations_df, n=3, name_col='name'):
    """Returns the n closest locations - useful for backup units."""
    df = locations_df.copy()
    df['distance_km'] = df.apply(
        lambda row: haversine(incident_lat, incident_lon, row['lat'], row['lon']),
        axis=1
    )
    top_n = df.nsmallest(n, 'distance_km')[[name_col, 'lat', 'lon', 'distance_km']]
    return top_n.reset_index(drop=True)


# Validation - Melbourne CBD test point 
TEST_LAT, TEST_LON = -37.8136, 144.9631

print("=" * 60)
print(f"  VALIDATION - Incident at Melbourne CBD ({TEST_LAT}, {TEST_LON})")
print("=" * 60)

for label, df in [('Emergency hospital', emergency_hospitals),
                  ('Fire station',       fire_stations),
                  ('Police station',     police_stations)]:
    result = find_nearest(TEST_LAT, TEST_LON, df)
    print(f"\n  Nearest {label}:")
    print(f"    {result['name']}  —  {result['distance_km']} km")

    top3 = find_nearest_n(TEST_LAT, TEST_LON, df, n=3)
    print(f"  Top 3:")
    for _, row in top3.iterrows():
        print(f"    {row['name']:<50} {row['distance_km']:.3f} km")

print("\n" + "=" * 60)

  VALIDATION - Incident at Melbourne CBD (-37.8136, 144.9631)

  Nearest Emergency hospital:
    St Vincent's Hospital  —  1.249 km
  Top 3:
    St Vincent's Hospital                              1.249 km
    The Royal Melbourne Hospital                       1.762 km
    St Vincent's Private Hospital East Melbourne       1.874 km

  Nearest Fire station:
    Fire Station No. 3  —  1.082 km
  Top 3:
    Fire Station No. 3                                 1.082 km
    Fire Station No. 2                                 1.125 km
    Fire Station No. 1                                 1.200 km

  Nearest Police station:
    Melbourne East Police Station  —  0.365 km
  Top 3:
    Melbourne East Police Station                      0.365 km
    Melbourne East Police Station                      0.435 km
    Melbourne Prosecutions                             0.450 km



**Results:** The `find_nearest()` function was validated using Melbourne CBD (−37.8136, 144.9631) as a test incident location. All three service types returned sensible nearest facilities with plausible distances:

- **Ambulance** - St Vincent's Hospital at 1.249 km, with The Royal Melbourne Hospital (1.762 km) and St Vincent's Private Hospital East Melbourne (1.874 km) as backup options. All three are within 2 km of the CBD, reflecting Melbourne's strong inner-city hospital coverage.
- **Fire** - Fire Station No. 3 at 1.082 km, with Stations No. 2 (1.125 km) and No. 1 (1.200 km) close behind. The tight clustering of the top 3 (all within ~120 metres of each other in distance ranking) suggests strong fire response coverage in the CBD.
- **Police** - Melbourne East Police Station at just 0.365 km, making it the closest facility of any service type. A duplicate entry appears in the top 3 at a slightly different distance (0.435 km), which may indicate two separate records for the same station or a nearby annex - worth investigating during data cleaning.

The Haversine-based lookup is functioning correctly across all three datasets and returning realistic distances for the Melbourne metropolitan area.

---
## 4 · Base Agent Class

All three agent types inherit from this base class, which provides shared functionality: availability tracking, assignment logging, and a standard `respond()` interface. This avoids duplicating common logic across the ambulance, fire, and police agents, and ensures every agent follows the same contract making the system easier to extend and maintain.

In [12]:
# Base Agent Class

class Agent:
    """
    Base class for all emergency response agents.
    Ambulance, Fire, and Police agents inherit from this.
    """

    def __init__(self, agent_id, agent_type, location="DEPOT"):
        self.agent_id   = agent_id
        self.agent_type = agent_type
        self.location   = location
        self.status     = "available"
        self.log        = []

    def is_available(self):
        return self.status == "available"

    def assign(self, emergency):
        if not self.is_available():
            return f"{self.agent_id} is currently busy and cannot respond."
        self.status = "busy"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} assigned to: {emergency}"
        self.log.append(entry)
        return entry

    def complete(self):
        self.status = "available"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} is now available again."
        self.log.append(entry)
        return entry

    def respond(self, emergency_type, location, lat=None, lon=None):
        return {
            "agent_id"       : self.agent_id,
            "agent_type"     : self.agent_type,
            "emergency_type" : emergency_type,
            "destination"    : location,
            "status"         : "dispatched",
        }

    def get_log(self):
        print(f"\n── Activity Log: {self.agent_id} ──")
        if not self.log:
            print("  No activity yet.")
        for entry in self.log:
            print(f"  {entry}")

    def __repr__(self):
        return f"<Agent {self.agent_id} | type={self.agent_type} | status={self.status}>"


# Base class validation
test_agent = Agent("TEST_01", "ambulance")

print("=" * 60)
print("  BASE AGENT CLASS - VALIDATION")
print("=" * 60)
print(f"  Created        : {test_agent}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Assigning...   : {test_agent.assign('cardiac arrest at Main St')}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Status         : {test_agent.status}")
print(f"  Completing...  : {test_agent.complete()}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Status         : {test_agent.status}")
test_agent.get_log()
print("=" * 60)

  BASE AGENT CLASS - VALIDATION
  Created        : <Agent TEST_01 | type=ambulance | status=available>
  Is available?  : True
  Assigning...   : [17:45:06] TEST_01 assigned to: cardiac arrest at Main St
  Is available?  : False
  Status         : busy
  Completing...  : [17:45:06] TEST_01 is now available again.
  Is available?  : True
  Status         : available

── Activity Log: TEST_01 ──
  [17:45:06] TEST_01 assigned to: cardiac arrest at Main St
  [17:45:06] TEST_01 is now available again.


**Results:** The base `Agent` class is functioning correctly. The validation demonstrates the full lifecycle of an agent: creation in `available` status, assignment to an incident (which transitions the agent to `busy` and logs a timestamped entry), and completion (which returns the agent to `available`). The activity log correctly records both events in chronological order, confirming that the logging mechanism works as expected for downstream audit trails.

---
## 5 · Specialised Agents - Ambulance, Fire, Police

Each agent extends the base class with domain-specific logic and uses `find_nearest()` to resolve the closest real facility from the datasets at dispatch time. This means every response reflects the actual nearest available unit.

| Agent | Dataset | Lookup |
|---|---|---|
| `AmbulanceAgent` | `emergency_hospitals` | Nearest emergency-capable hospital |
| `FireAgent` | `fire_stations` | Nearest CFA / MFB station |
| `PoliceAgent` | `police_stations` | Nearest Victoria Police station |

In [15]:

# Ambulance Agent


class AmbulanceAgent(Agent):
    """
    Handles medical emergencies.
    Resolves nearest emergency hospital from real dataset.
    """

    ALS_REQUIRED = ["cardiac_arrest", "car_accident_major", "building_collapse", "drowning"]

    def __init__(self, agent_id, location="DEPOT_AMBULANCE"):
        super().__init__(agent_id, agent_type="ambulance", location=location)
        self.crew_size = 2

    def assess_severity(self, emergency_type):
        return "ALS" if emergency_type in self.ALS_REQUIRED else "BLS"

    def get_crew_size(self, care_level):
        return 3 if care_level == "ALS" else 2

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        care_level = self.assess_severity(emergency_type)
        crew       = self.get_crew_size(care_level)

        if lat is not None and lon is not None:
            hospital_info    = find_nearest(lat, lon, emergency_hospitals)
            nearest_hospital = hospital_info['name']
            hospital_dist    = f"{hospital_info['distance_km']} km"
        else:
            nearest_hospital = "unknown (no coordinates provided)"
            hospital_dist    = "unknown"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"          : self.agent_id,
            "agent_type"        : "ambulance",
            "emergency_type"    : emergency_type,
            "destination"       : location,
            "coordinates"       : f"{lat}, {lon}" if lat else "not provided",
            "care_level"        : care_level,
            "crew_size"         : crew,
            "nearest_hospital"  : nearest_hospital,
            "hospital_distance" : hospital_dist,
            "status"            : "dispatched",
            "route"             : "PENDING — routing engine (Sprint 2)"
        }


# Fire Agent

class FireAgent(Agent):
    """
    Handles fire and hazardous material emergencies.
    Resolves nearest fire station from real dataset.
    """

    EQUIPMENT_MAP = {
        "fire"              : ["hose_lines", "breathing_apparatus", "ladder_truck"],
        "car_accident_major": ["hydraulic_rescue_tools", "hose_lines"],
        "building_collapse" : ["search_rescue_kit", "breathing_apparatus", "hose_lines"],
        "gas_leak"          : ["hazmat_suit", "gas_detector", "breathing_apparatus"],
    }

    TRUCKS_NEEDED = {
        "fire": 2, "car_accident_major": 1, "building_collapse": 3, "gas_leak": 1,
    }

    def __init__(self, agent_id, location="DEPOT_FIRE"):
        super().__init__(agent_id, agent_type="fire", location=location)

    def get_equipment(self, emergency_type):
        return self.EQUIPMENT_MAP.get(emergency_type, ["standard_kit"])

    def get_trucks_needed(self, emergency_type):
        return self.TRUCKS_NEEDED.get(emergency_type, 1)

    def is_hazmat(self, emergency_type):
        return emergency_type == "gas_leak"

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        equipment = self.get_equipment(emergency_type)
        trucks    = self.get_trucks_needed(emergency_type)
        hazmat    = self.is_hazmat(emergency_type)

        if lat is not None and lon is not None:
            station_info    = find_nearest(lat, lon, fire_stations)
            nearest_station = station_info['name']
            station_dist    = f"{station_info['distance_km']} km"
        else:
            nearest_station = "unknown (no coordinates provided)"
            station_dist    = "unknown"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"         : self.agent_id,
            "agent_type"       : "fire",
            "emergency_type"   : emergency_type,
            "destination"      : location,
            "coordinates"      : f"{lat}, {lon}" if lat else "not provided",
            "trucks_dispatch"  : trucks,
            "equipment"        : equipment,
            "hazmat_situation" : hazmat,
            "nearest_station"  : nearest_station,
            "station_distance" : station_dist,
            "status"           : "dispatched",
            "route"            : "PENDING — routing engine (Sprint 2)"
        }


# Police Agent

class PoliceAgent(Agent):
    """
    Handles crime, crowd control, and scene security.
    Default first responder for unknown emergencies.
    Resolves nearest police station from real dataset.
    """

    ARMED_RESPONSE_REQUIRED = ["robbery", "assault", "building_collapse", "unknown"]
    UNITS_NEEDED = {
        "robbery": 2, "assault": 2, "car_accident_minor": 1,
        "car_accident_major": 2, "building_collapse": 3,
        "gas_leak": 2, "drowning": 1, "unknown": 1,
    }
    CORDON_REQUIRED = ["car_accident_major", "building_collapse", "gas_leak", "fire"]

    def __init__(self, agent_id, location="DEPOT_POLICE"):
        super().__init__(agent_id, agent_type="police", location=location)
        self.is_lead = False

    def needs_armed_response(self, emergency_type):
        return emergency_type in self.ARMED_RESPONSE_REQUIRED

    def get_units_needed(self, emergency_type):
        return self.UNITS_NEEDED.get(emergency_type, 1)

    def needs_cordon(self, emergency_type):
        return emergency_type in self.CORDON_REQUIRED

    def set_as_lead(self):
        self.is_lead = True

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        armed_response = self.needs_armed_response(emergency_type)
        units          = self.get_units_needed(emergency_type)
        cordon         = self.needs_cordon(emergency_type)

        if lat is not None and lon is not None:
            station_info    = find_nearest(lat, lon, police_stations)
            nearest_station = station_info['name']
            station_dist    = f"{station_info['distance_km']} km"
        else:
            nearest_station = "unknown (no coordinates provided)"
            station_dist    = "unknown"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"         : self.agent_id,
            "agent_type"       : "police",
            "emergency_type"   : emergency_type,
            "destination"      : location,
            "coordinates"      : f"{lat}, {lon}" if lat else "not provided",
            "units_dispatch"   : units,
            "armed_response"   : armed_response,
            "cordon_required"  : cordon,
            "lead_coordinator" : self.is_lead,
            "nearest_station"  : nearest_station,
            "station_distance" : station_dist,
            "status"           : "dispatched",
            "route"            : "PENDING — routing engine (Sprint 2)"
        }


# Test all three agents 
TEST_LAT, TEST_LON = -37.8136, 144.9631

amb  = AmbulanceAgent("AMB_TEST")
fire = FireAgent("FIRE_TEST")
pol  = PoliceAgent("POL_TEST")

test_cases = [
    (amb,  "cardiac_arrest",     "Flinders St Station"),
    (amb,  "car_accident_minor", "St Kilda Road"),
    (fire, "fire",               "Queen Victoria Market"),
    (fire, "gas_leak",           "South Melbourne"),
    (pol,  "robbery",            "Bourke Street Mall"),
    (pol,  "car_accident_major", "West Gate Freeway"),
]

print("=" * 65)
print("  SPECIALISED AGENTS - VALIDATION")
print("  Test location: Melbourne CBD (-37.8136, 144.9631)")
print("=" * 65)

for agent, etype, loc in test_cases:
    agent.status = "available"
    agent.log    = []
    if hasattr(agent, 'is_lead'):
        agent.is_lead = False

    r = agent.respond(etype, loc, lat=TEST_LAT, lon=TEST_LON)
    print(f"\n  [{r['agent_type'].upper()}] {r['agent_id']} — {etype}")
    for k, v in r.items():
        if k not in ['agent_id', 'agent_type']:
            print(f"    {k:<24} : {v}")

print("\n" + "=" * 65)

  SPECIALISED AGENTS - VALIDATION
  Test location: Melbourne CBD (-37.8136, 144.9631)

  [AMBULANCE] AMB_TEST — cardiac_arrest
    emergency_type           : cardiac_arrest
    destination              : Flinders St Station
    coordinates              : -37.8136, 144.9631
    care_level               : ALS
    crew_size                : 3
    nearest_hospital         : St Vincent's Hospital
    hospital_distance        : 1.249 km
    status                   : dispatched
    route                    : PENDING — routing engine (Sprint 2)

  [AMBULANCE] AMB_TEST — car_accident_minor
    emergency_type           : car_accident_minor
    destination              : St Kilda Road
    coordinates              : -37.8136, 144.9631
    care_level               : BLS
    crew_size                : 2
    nearest_hospital         : St Vincent's Hospital
    hospital_distance        : 1.249 km
    status                   : dispatched
    route                    : PENDING — routing engine (Sprint

**Results:** All three specialised agents were validated across six test scenarios using Melbourne CBD as the incident location.

**Ambulance agent** correctly differentiated between severity levels - cardiac arrest triggered Advanced Life Support (ALS) with a crew of 3, while a minor car accident triggered Basic Life Support (BLS) with a crew of 2. Both dispatches resolved St Vincent's Hospital as the nearest emergency-capable facility at 1.249 km, which is consistent with the `find_nearest()` validation in Section 3.

**Fire agent** adapted its response based on emergency type. A structural fire dispatched 2 trucks with standard firefighting equipment (hose lines, breathing apparatus, ladder truck) and no hazmat flag. A gas leak dispatched 1 truck with specialised hazmat gear (hazmat suit, gas detector, breathing apparatus) and correctly flagged `hazmat_situation: True`. Both resolved Fire Station No. 3 at 1.082 km.

**Police agent** correctly toggled between armed and unarmed responses - robbery triggered `armed_response: True` with 2 units, while a major car accident triggered `armed_response: False` but flagged `cordon_required: True` with 2 units. Both resolved Melbourne East Police Station at 0.365 km.

All six test cases returned real facility names and distances from the Victorian datasets, confirming that the placeholder data from Sprint 1 has been fully replaced.

---
## 6 · Central Dispatch Function

The entry point for the entire system. Given an emergency type, location, and coordinates, `dispatch()`:

1. Looks up the dispatch table to determine which agents are needed
2. Finds an available unit of each required type from the agent pool
3. Sets police as lead coordinator in multi-agent responses
4. Calls `respond()` on each agent (which triggers real nearest-unit lookups)
5. Returns a structured incident report

This is the function that LLM integration layer will call directly- will be passed in the extracted emergency type and location, and `dispatch()` handles everything from there.

In [18]:

# Agent Pool & Dispatch Function

AGENT_POOL = {
    "ambulance": [AmbulanceAgent("AMB_01"), AmbulanceAgent("AMB_02")],
    "fire"     : [FireAgent("FIRE_01"),     FireAgent("FIRE_02")],
    "police"   : [PoliceAgent("POL_01"),    PoliceAgent("POL_02")],
}

def reset_agent_pool():
    """Reset all agents to available with clean logs."""
    for agents in AGENT_POOL.values():
        for agent in agents:
            agent.status = "available"
            agent.log    = []
            if hasattr(agent, 'is_lead'):
                agent.is_lead = False


def dispatch(emergency_type, location, lat=None, lon=None):
    """
    Main dispatch function.

    Parameters
    ----------
    emergency_type : str
        Must match a key in DISPATCH_TABLE (falls back to 'unknown').
    location : str
        Human-readable location description.
    lat, lon : float, optional
        Incident coordinates for real nearest-unit lookups.

    Returns
    
    dict : Full incident report with all agent responses.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if emergency_type not in DISPATCH_TABLE:
        emergency_type = "unknown"

    dispatch_info = DISPATCH_TABLE[emergency_type]
    agents_needed = dispatch_info["agents"]
    priority      = dispatch_info["priority"]
    notes         = dispatch_info["notes"]

    responses   = []
    agents_sent = []
    unavailable = []

    for agent_type in agents_needed:
        agent = None
        for a in AGENT_POOL[agent_type]:
            if a.is_available():
                agent = a
                break

        if agent is None:
            unavailable.append(agent_type)
            continue

        if agent_type == "police" and len(agents_needed) > 1:
            agent.set_as_lead()

        response = agent.respond(emergency_type, location, lat=lat, lon=lon)
        responses.append(response)
        agents_sent.append(agent.agent_id)

    return {
        "incident_id"     : f"INC_{datetime.now().strftime('%H%M%S')}",
        "timestamp"       : timestamp,
        "emergency_type"  : emergency_type,
        "location"        : location,
        "coordinates"     : f"{lat}, {lon}" if lat else "not provided",
        "priority"        : priority,
        "dispatch_notes"  : notes,
        "agents_sent"     : agents_sent,
        "unavailable"     : unavailable if unavailable else "none",
        "total_responses" : len(responses),
        "responses"       : responses,
    }


def print_incident_report(report):
    """print a full incident report."""
    print("\n" + "=" * 65)
    print(f"  INCIDENT REPORT — {report['incident_id']}")
    print("=" * 65)
    print(f"  Timestamp      : {report['timestamp']}")
    print(f"  Emergency      : {report['emergency_type'].upper()}")
    print(f"  Location       : {report['location']}")
    print(f"  Coordinates    : {report['coordinates']}")
    print(f"  Priority       : {report['priority']} (1 = critical)")
    print(f"  Notes          : {report['dispatch_notes']}")
    print(f"  Agents sent    : {', '.join(report['agents_sent'])}")
    print(f"  Unavailable    : {report['unavailable']}")
    print(f"  Total responses: {report['total_responses']}")
    print("\n  ── Agent Responses ──")
    for r in report["responses"]:
        print(f"\n  [{r['agent_type'].upper()}] {r['agent_id']}")
        for key, value in r.items():
            if key not in ["agent_id", "agent_type"]:
                print(f"    {key:<24} : {value}")
    print("=" * 65)


# Dispatch validation — 3 scenarios 
CBD_LAT, CBD_LON = -37.8136, 144.9631

print("=" * 65)
print("  DISPATCH FUNCTION — VALIDATION")
print("  3 scenarios: single agent, two agents, all three agents")
print("=" * 65)

# Single agent - robbery (police only)
reset_agent_pool()
print_incident_report(dispatch("robbery", "Bourke Street Mall", lat=CBD_LAT, lon=CBD_LON))

# Two agents - assault (police + ambulance)
reset_agent_pool()
print_incident_report(dispatch("assault", "Federation Square", lat=CBD_LAT, lon=CBD_LON))

# All three - building collapse (fire + ambulance + police)
reset_agent_pool()
print_incident_report(dispatch("building_collapse", "Docklands", lat=CBD_LAT, lon=CBD_LON))

  DISPATCH FUNCTION — VALIDATION
  3 scenarios: single agent, two agents, all three agents

  INCIDENT REPORT — INC_174507
  Timestamp      : 2026-04-03 17:45:07
  Emergency      : ROBBERY
  Location       : Bourke Street Mall
  Coordinates    : -37.8136, 144.9631
  Priority       : 2 (1 = critical)
  Notes          : Police only
  Agents sent    : POL_01
  Unavailable    : none
  Total responses: 1

  ── Agent Responses ──

  [POLICE] POL_01
    emergency_type           : robbery
    destination              : Bourke Street Mall
    coordinates              : -37.8136, 144.9631
    units_dispatch           : 2
    armed_response           : True
    cordon_required          : False
    lead_coordinator         : False
    nearest_station          : Melbourne East Police Station
    station_distance         : 0.365 km
    status                   : dispatched
    route                    : PENDING — routing engine (Sprint 2)

  INCIDENT REPORT — INC_174507
  Timestamp      : 2026-04-03

**Results:** The dispatch function was validated across three scenarios of increasing complexity, all using Melbourne CBD coordinates.

**Single-agent dispatch (robbery)** - correctly dispatched only POL_01 with armed response enabled and 2 units. The `lead_coordinator` field is `False` because police are only designated as lead when multiple service types respond together. Melbourne East Police Station was resolved at 0.365 km.

**Two-agent dispatch (assault)** - correctly dispatched both POL_01 and AMB_01. Police was automatically designated as lead coordinator (`lead_coordinator: True`) since multiple services were involved. The ambulance assessed the assault as BLS (Basic Life Support) with a crew of 2, which is appropriate for a non-critical injury. Both agents resolved their nearest real facilities independently.

**Three-agent dispatch (building collapse)** - the most complex scenario, correctly mobilising all three services. Fire dispatched 3 trucks with search and rescue equipment, ambulance escalated to ALS with a crew of 3 (reflecting the critical nature of a collapse), and police dispatched 3 units with both armed response and scene cordoning enabled. Police was again correctly designated as lead coordinator. All three agents resolved real nearest facilities from the Victorian datasets, with no agents listed as unavailable.

Across all three scenarios, the dispatch function correctly scaled agent combinations, priority levels, and domain-specific logic based on the emergency type confirming the dispatch table, agent pool, and nearest-unit lookups are working together as intended.

---
## 7 · Real Crash Incidents through Dispatch

This test pulls real incidents directly from the Victorian crash dataset and runs them through the full dispatch pipeline. Crash severity is mapped to the system's emergency types using a simple rule-based function - this mapping will eventually be replaced by LLM classification layer, which will extract emergency types from natural language caller input.

| Crash Severity | Mapped Emergency Type | Agents Dispatched |
|---|---|---|
| Fatal | `car_accident_major` | Ambulance + Fire + Police |
| Serious injury | `car_accident_major` | Ambulance + Fire + Police |
| Struck pedestrian | `car_accident_major` | Ambulance + Fire + Police |
| Non-injury | `car_accident_minor` | Ambulance + Police |

In [21]:

# Real Crash Incidents through Dispatch

def map_severity_to_emergency(row):
    """Maps crash dataset severity to dispatch emergency types."""
    severity = row['severity'].lower()
    inc_type = row['incident_type'].lower()

    if 'fatal' in severity:
        return 'car_accident_major'
    elif 'serious' in severity:
        return 'car_accident_major'
    elif 'struck pedestrian' in inc_type:
        return 'car_accident_major'
    else:
        return 'car_accident_minor'


# Sample 5 real incidents
fatal   = crash_data[crash_data['severity'] == 'Fatal accident'].sample(2, random_state=42)
serious = crash_data[crash_data['severity'] == 'Serious injury accident'].sample(2, random_state=42)
minor   = crash_data[crash_data['severity'] == 'Non injury accident'].sample(1, random_state=42)
sample  = pd.concat([fatal, serious, minor]).reset_index(drop=True)

print("=" * 65)
print("  REAL VICTORIAN CRASH INCIDENTS - DISPATCH TEST")
print("  5 incidents sampled across fatal, serious, and non-injury")
print("=" * 65)

for _, row in sample.iterrows():
    reset_agent_pool()

    emergency_type = map_severity_to_emergency(row)
    location       = f"Incident {row['incident_id']} ({row['incident_type']})"
    lat, lon       = row['lat'], row['lon']

    print(f"\n  ┌─ Source Record ─────────────────────────────────────")
    print(f"  │ ID       : {row['incident_id']}")
    print(f"  │ Date     : {row['date']}  Time: {row['time']}")
    print(f"  │ Type     : {row['incident_type']}")
    print(f"  │ Severity : {row['severity']}")
    print(f"  │ People   : {row['people_involved']}")
    print(f"  │ Coords   : {lat}, {lon}")
    print(f"  │ Mapped   : {emergency_type}")
    print(f"  └──────────────────────────────────────────────────────")

    report = dispatch(emergency_type, location, lat=lat, lon=lon)
    print_incident_report(report)

  REAL VICTORIAN CRASH INCIDENTS - DISPATCH TEST
  5 incidents sampled across fatal, serious, and non-injury

  ┌─ Source Record ─────────────────────────────────────
  │ ID       : T20210019823
  │ Date     : 2021-09-25  Time: 22:25:00
  │ Type     : Collision with a fixed object
  │ Severity : Fatal accident
  │ People   : 1
  │ Coords   : -36.71361, 146.41008
  │ Mapped   : car_accident_major
  └──────────────────────────────────────────────────────

  INCIDENT REPORT — INC_174508
  Timestamp      : 2026-04-03 17:45:08
  Emergency      : CAR_ACCIDENT_MAJOR
  Location       : Incident T20210019823 (Collision with a fixed object)
  Coordinates    : -36.71361, 146.41008
  Priority       : 1 (1 = critical)
  Notes          : All services — entrapment likely, major injuries expected
  Agents sent    : AMB_01, FIRE_01, POL_01
  Unavailable    : none
  Total responses: 3

  ── Agent Responses ──

  [AMBULANCE] AMB_01
    emergency_type           : car_accident_major
    destination        

**Results:** Five real Victorian crash incidents were successfully dispatched - 2 fatal, 2 serious injury, and 1 non-injury spanning locations from regional Victoria to inner Melbourne.

**Fatal incidents** both mapped correctly to `car_accident_major` (priority 1) and triggered all three services. The first incident (T20210019823) occurred in regional Victoria near Wangaratta, which exposed an important finding: the nearest facilities were over 130 km away (Maroondah Hospital at 158.9 km, Healesville CFA at 131.3 km, Warburton Police Station at 132.2 km). This is expected given that the facility datasets are concentrated in the greater Melbourne area, and highlights the need for expanded regional coverage in future dataset iterations. The second fatal incident (T20160002505) occurred closer to Melbourne, with all facilities resolving within 6.3 km - a much more realistic response scenario.

**Serious injury incidents** also mapped to `car_accident_major` and dispatched all three services. Distances were reasonable - the closest facility across both incidents was Fire Station No. 12 at just 1.41 km. Both correctly triggered ALS care level with a crew of 3, scene cordoning, and police as lead coordinator.

**Non-injury incident** (T20220009177) mapped to `car_accident_minor` (priority 2) and dispatched only ambulance and police - no fire service, which is correct. The ambulance downgraded to BLS with a crew of 2, and police sent just 1 unit with no armed response or cordoning. Notably, Fitzroy Police Station resolved at only 0.131 km, and St Vincent's Hospital at 0.673 km, reflecting excellent inner-city coverage.

One data quality note: incident T20190004503 resolved to an "Unnamed Police Facility" at 2.463 km, suggesting some records in `police_stations.csv` have missing or incomplete name fields worth flagging for the dataset team to review.

All 5 incidents dispatched with zero failures and zero unavailable agents.

---
## 8 · Full System Test - All Emergency Types

Runs all 10 emergency types through dispatch using the Melbourne CBD as a common test point. Each scenario resets the agent pool to simulate independent incidents. This confirms that every emergency type in the dispatch table produces a valid response with the correct agent combination and priority level.

In [24]:

# Full System Test — All 10 Emergency Types

# Melbourne CBD as common test location
CBD_LAT, CBD_LON = -37.8136, 144.9631

test_scenarios = [
    ("cardiac_arrest",      "Flinders St Station"),
    ("fire",                "Queen Victoria Market"),
    ("car_accident_minor",  "St Kilda Road"),
    ("car_accident_major",  "West Gate Freeway"),
    ("robbery",             "Bourke Street Mall"),
    ("assault",             "Federation Square"),
    ("building_collapse",   "Docklands"),
    ("gas_leak",            "South Melbourne"),
    ("drowning",            "St Kilda Beach"),
    ("unknown",             "Carlton Gardens"),
]

print("=" * 75)
print("  FULL SYSTEM TEST - ALL 10 EMERGENCY TYPES")
print("  Location: Melbourne CBD area")
print("=" * 75)
print(f"  {'EMERGENCY':<25} {'PRI':<5} {'AGENTS DISPATCHED':<35} {'STATUS'}")
print("  " + "─" * 71)

summary = []
for emergency_type, location in test_scenarios:
    reset_agent_pool()
    report     = dispatch(emergency_type, location, lat=CBD_LAT, lon=CBD_LON)
    agents_str = ", ".join(report["agents_sent"]) if report["agents_sent"] else "NONE"
    status     = "OK" if report["total_responses"] > 0 else "FAILED"
    summary.append((emergency_type, report["priority"], agents_str, status))
    print(f"  {emergency_type:<25} {report['priority']:<5} {agents_str:<35} {status}")

total    = len(summary)
ok       = sum(1 for s in summary if s[3] == "OK")
failed   = total - ok
critical = sum(1 for s in summary if s[1] == 1)

print("  " + "─" * 71)
print(f"\n  Total scenarios         : {total}")
print(f"  Successfully dispatched : {ok}/{total}")
print(f"  Failed dispatches       : {failed}")
print(f"  Critical (P1) incidents : {critical}")
print("=" * 75)

  FULL SYSTEM TEST - ALL 10 EMERGENCY TYPES
  Location: Melbourne CBD area
  EMERGENCY                 PRI   AGENTS DISPATCHED                   STATUS
  ───────────────────────────────────────────────────────────────────────
  cardiac_arrest            1     AMB_01                              OK
  fire                      1     FIRE_01, AMB_01                     OK
  car_accident_minor        2     AMB_01, POL_01                      OK
  car_accident_major        1     AMB_01, FIRE_01, POL_01             OK
  robbery                   2     POL_01                              OK
  assault                   2     POL_01, AMB_01                      OK
  building_collapse         1     FIRE_01, AMB_01, POL_01             OK
  gas_leak                  1     FIRE_01, POL_01                     OK
  drowning                  1     AMB_01, POL_01                      OK
  unknown                   3     POL_01                              OK
  ──────────────────────────────────────────

**Results:** All 10 emergency types dispatched successfully with zero failures.

The agent combinations match the dispatch table exactly:
- **Single-agent responses** - cardiac arrest (ambulance only), robbery (police only), and unknown (police only) each dispatched one service as expected.
- **Two-agent responses** - fire (fire + ambulance), minor car accident (ambulance + police), assault (police + ambulance), gas leak (fire + police), and drowning (ambulance + police) each dispatched the correct pair.
- **Three-agent responses** - major car accident and building collapse both mobilised all three services (ambulance, fire, police).

Priority levels are correctly assigned: 6 incidents at priority 1 (critical), 3 at priority 2 (high), and 1 at priority 3 (medium). The `unknown` type correctly defaults to the lowest priority with police-only response, acting as the system's catch-all for unrecognised emergency types.

10/10 scenarios passed with 0 failed dispatches, confirming full coverage of the dispatch table against real Victorian facility data.

---
## 9 · Summary and Conclusion

### Datasets Integrated

| Dataset | Records | Used By |
|---|---|---|
| `hospitals.csv` | 26 emergency-capable | `AmbulanceAgent` - nearest hospital by Haversine distance |
| `fire_stations.csv` | 205 stations | `FireAgent` - nearest fire station by Haversine distance |
| `police_stations.csv` | 124 stations | `PoliceAgent` - nearest police station by Haversine distance |
| `emergency_crash.csv` | 194,352 incidents | Real Victorian crash data for dispatch validation |
| `road_nodes.csv` | 228,213 nodes | Road network graph - routing engine handoff |
| `road_edges.csv` | 501,205 edges | Road network graph - routing engine handoff |
| `cleaned_transport_2025 (2).csv` | 258,573 records | Public transport network data |
| `pedestrian_data.csv` | 734,064 records | Pedestrian activity and congestion data |

### Test Results

| Metric | Result |
|---|---|
| Emergency types covered | 10/10 |
| Full system test pass rate | 10/10 |
| Real crash incidents dispatched | 5/5 |
| Failed dispatches | 0 |
| All three service datasets live |  Hospitals, Fire Stations, Police Stations |
| Nearest-unit lookup validated |  Haversine against all 3 facility datasets |

### Key Observations

- **Inner-city coverage is strong.** For incidents within the Melbourne metropolitan area, all three services resolved nearest facilities within 1–7 km, with police stations often the closest (as low as 0.131 km in Fitzroy).
- **Regional coverage is limited.** The facility datasets are concentrated in greater Melbourne. A fatal incident near Wangaratta resolved facilities over 130 km away, highlighting the need for expanded regional station data in future iterations.
- **Data quality flag.** One police station record resolved as "Unnamed Police Facility", suggesting incomplete name fields in `police_stations.csv`. This has been noted for the dataset team to review.
- **Severity mapping works but is temporary.** The rule-based `map_severity_to_emergency()` function correctly classified all 5 sampled incidents. This will be replaced by LLM classification layer, which will extract emergency types directly from natural language caller input.

### Pending
| Item | Notes |
|---|---|
| `route: PENDING` - real routing | `road_nodes.csv` and `road_edges.csv` are loaded and ready for graph construction |
| `map_severity_to_emergency()` - LLM classification | Current rule-based mapping to be replaced with natural language extraction |
| Expanded unit pool (>2 per type) | Current pool of 2 units per service type is sufficient for testing but not for production |
| Regional facility coverage | Additional station data needed for areas outside greater Melbourne |
| Transport and pedestrian data integration | Both datasets loaded and validated - integration into dispatch logic to follow |

### Conclusion

The Agent Dispatch Module is fully operational on real Victorian data. All placeholder values from Sprint 1 have been replaced with verified geographic datasets - every agent now resolves its nearest facility dynamically using Haversine distance calculations at dispatch time. The system was validated across all 10 emergency types with a 100% dispatch success rate, and further tested against 5 real crash incidents spanning fatal, serious injury, and non-injury severities.

The `dispatch()` function provides a clean, single-entry-point interface that accepts an emergency type, location, and coordinates and returns a structured incident report. This interface is ready for direct integration with bothLLM layer (which will replace the temporary severity mapping) and routing engine (which will populate the currently pending route fields using the loaded road network data).

Eight datasets totalling over 1.9 million records have been loaded and validated, with the three core facility datasets (hospitals, fire stations, police stations) actively powering the dispatch logic and the remaining five (crash data, road network, transport, pedestrian) ready for downstream integration in Sprint 2.
